# Native Curve CryptoSwap v2.1 vs Dusk current DH2

Reproducible companion to `DECISION_REPORT.md`. It reads only committed frozen inputs and benchmark outputs. Curve is an independent clean-room analytical reference pinned in `CURVE_REFERENCE.md`; no Curve source is vendored. Historical prices are observed SOLUSDT paths, while user order flow is simulated.

In [ ]:
from pathlib import Path
import csv, hashlib, json, math, statistics, sys

ROOT = Path.cwd()
if not (ROOT / 'results').exists():
    ROOT = Path.cwd() / 'analysis' / 'cryptoswap-vs-dh2'
RESULTS = ROOT / 'results'
sys.path.insert(0, str(ROOT))
from native_validation import cpmm_product_selection_is_consistent, dusk_product_selection_is_consistent, native_decision_is_consistent
summary = json.loads((RESULTS / 'native_summary.json').read_text())
assert summary['status'] == 'complete'
assert summary['validation']['canonical_output'] is True
assert native_decision_is_consistent(summary['profile_selection_policy'], summary['decision'])
assert dusk_product_selection_is_consistent(summary['dusk_product_selection_policy'])
assert cpmm_product_selection_is_consistent(summary['cpmm_product_selection_policy'])
assert set(summary['profile_selection']) == {'curve_native_v2_1', 'dusk_native', 'dusk_cpmm'}
COUNT_FILES = {
    'profile_rows': 'native_profile_selection.csv',
    'training_rows': 'native_training_results.csv',
    'held_out_rows': 'native_held_out.csv',
    'paired_rows': 'native_paired.csv',
    'cpmm_rows': 'native_cpmm_control.csv',
    'cpmm_matched_policy_ablation_rows': 'native_cpmm_matched_policy_ablation.csv',
    'counterfactual_rows': 'native_controller_counterfactual.csv',
    'neighborhood_rows': 'native_neighborhood.csv',
    'scale_rows': 'native_scale_sensitivity.csv',
    'cadence_rows': 'native_cadence_sensitivity.csv',
    'basis_sensitivity_rows': 'native_matching_basis_sensitivity.csv',
    'fee_split_sensitivity_rows': 'native_dusk_fee_split_sensitivity.csv',
}
CANONICAL_COUNTS = {
    'profile_rows': 32, 'training_rows': 4096, 'held_out_rows': 480,
    'paired_rows': 240, 'cpmm_rows': 240, 'cpmm_matched_policy_ablation_rows': 240, 'counterfactual_rows': 480,
    'neighborhood_rows': 1008, 'scale_rows': 288, 'cadence_rows': 96,
    'basis_sensitivity_rows': 1440, 'fee_split_sensitivity_rows': 960,
}
expected_counts = summary['validation']['expected_counts']
assert expected_counts == CANONICAL_COUNTS
assert set(expected_counts) == set(COUNT_FILES)
for field, filename in COUNT_FILES.items():
    with (RESULTS / filename).open() as source:
        actual = sum(1 for _ in csv.DictReader(source))
    assert actual == int(expected_counts[field]), (field, actual, expected_counts[field])
    if field != 'training_rows':
        assert int(summary['validation'][field]) == actual
AUDIT_FILES = [filename for field, filename in COUNT_FILES.items() if field not in {'profile_rows', 'paired_rows', 'fee_split_sensitivity_rows'}]
audit_rows = []
for filename in AUDIT_FILES:
    with (RESULTS / filename).open() as source:
        audit_rows.extend(csv.DictReader(source))
accounting_errors = [abs(float(row['accounting_identity_error'])) for row in audit_rows]
assert all(math.isfinite(value) for value in accounting_errors)
assert max(accounting_errors) <= 1e-5
decision_fields = ('net_lp_return_after_protocol_fee_bps', 'net_lp_return_vs_hold_bps', 'captured_user_volume_fraction', 'selected_execution_cost_bps', 'depth_degradation_fraction', 'marginal_tracking_stale_fraction', 'accounting_identity_error')
assert all(math.isfinite(float(row[field])) for row in audit_rows for field in decision_fields)
assert summary['validation']['all_decision_metrics_finite'] is True
assert max(int(row['controller_external_price_reads']) for row in audit_rows) == 0
assert sum(int(row['arbitrage_capped_event_count']) for row in audit_rows) == 0
assert summary['validation']['arbitrage_capped_event_count'] == 0
assert summary['validation']['scored_arbitrage_capped_event_count'] == 0
assert summary['validation']['arbitrage_numerical_failure_count'] == 0
assert summary['validation']['scored_arbitrage_numerical_failure_count'] == 0
assert summary['validation']['arbitrage_cap_counter_scope'] == 'all_periods_including_burn_in_and_score'
post_run_audit = summary['validation']['post_run_audit']
assert post_run_audit['status'] == 'passed'
assert post_run_audit['scope'] == 'all_underlying_replay_matrices'
assert int(post_run_audit['audited_rows']) == len(audit_rows) == 8368
def percentile(values, q):
    ordered = sorted(values)
    position = (len(ordered) - 1) * q
    lower = int(math.floor(position))
    upper = int(math.ceil(position))
    if lower == upper:
        return ordered[lower]
    weight = position - lower
    return ordered[lower] * (1.0 - weight) + ordered[upper] * weight
summary['reference'], summary['validation']

## Frozen source identity

In [ ]:
metadata = json.loads((ROOT / 'data/native_history_metadata.json').read_text())
history_path = ROOT / 'data/solusdt_native_windows.csv'
history_sha = hashlib.sha256(history_path.read_bytes()).hexdigest()
assert history_sha == metadata['snapshot_sha256']
{'dataset': history_path.name, 'sha256': history_sha, 'windows': len(metadata['windows'])}

## Separate synthetic-only candidate and CPMM locks

In [ ]:
selection_policy = summary['profile_selection_policy']
dusk_selection_policy = summary['dusk_product_selection_policy']
cpmm_selection_policy = summary['cpmm_product_selection_policy']
{
    'selected_profiles': summary['profile_selection'],
    'common_service_relaxation': selection_policy['common_service_relaxation'],
    'comparison_pair_ready_on_training_service': selection_policy['comparison_pair_ready_on_training_service'],
    'minimum_required_relaxation_by_system': selection_policy['minimum_required_relaxation_by_system'],
    'dusk_product_service_relaxation': dusk_selection_policy['service_relaxation'],
    'dusk_product_ready_on_training_service': dusk_selection_policy['product_ready_on_training_service'],
    'dusk_readiness_is_independent': dusk_selection_policy['independent_from_common_curve_dh2_relaxation'],
    'cpmm_service_relaxation': cpmm_selection_policy['service_relaxation'],
    'cpmm_product_ready_on_training_service': cpmm_selection_policy['product_ready_on_training_service'],
    'cpmm_contaminates_candidate_selection': cpmm_selection_policy['contaminates_concentrated_candidate_selection'],
}

## Headline held-out economics

Positive deltas favor Dusk. Gross returns describe the same profiles selected on fee-adjusted synthetic return; they are diagnostic, not a separate universal ranking. Fee-adjusted returns apply the declared Curve admin liability and Dusk base-fee revenue share.

In [ ]:
headline_fields = [
    'paired_cells', 'median_dusk_minus_curve_gross_return_bps',
    'median_dusk_minus_curve_return_after_protocol_fee_bps',
    'p10_dusk_minus_curve_return_after_protocol_fee_bps',
    'window_median_win_fraction',
    'median_dusk_minus_curve_selected_execution_cost_bps',
    'median_dusk_minus_curve_captured_volume_fraction',
]
{route: {field: values[field] for field in headline_fields} for route, values in summary['held_out'].items()}

In [ ]:
with (RESULTS / 'native_paired.csv').open() as source:
    paired = list(csv.DictReader(source))
paired_distribution = {}
for route in ('fixed', 'elastic'):
    for flow in ('benign', 'mixed', 'toxic_delayed', 'low_volume_stuck'):
        values = sorted(float(row['dusk_minus_curve_net_lp_return_after_protocol_fee_bps']) for row in paired if row['routing_mode'] == route and row['flow_regime'] == flow)
        paired_distribution[f'{flow}:{route}'] = {
            'cells': len(values), 'median_bps': statistics.median(values),
            'p10_bps': percentile(values, .10),
            'win_fraction': sum(value > 0 for value in values) / len(values),
        }
paired_distribution

## Robustness gates by flow, route, and matching basis

In [ ]:
dusk_decision = summary['decision']['fee_adjusted']['dusk']
dict(sorted(dusk_decision['by_flow_and_route'].items(), key=lambda item: item[1]['median_return_advantage_bps']))

In [ ]:
dict(sorted(dusk_decision['by_matching_basis_flow_and_route'].items(), key=lambda item: item[1]['median_return_advantage_bps']))

## Dusk base-fee revenue-share sensitivity

This is exact revaluation, not a second simulation: changing the manager/protocol share of the base fee changes LP ownership accounting but not quotes, reserves, surcharge retention, or center moves.

In [ ]:
with (RESULTS / 'native_dusk_fee_split_sensitivity.csv').open() as source:
    fee_split = list(csv.DictReader(source))
fee_table = {}
for share in (0, 500, 2500, 10000):
    for route in ('fixed', 'elastic'):
        values = sorted(float(row['dusk_minus_curve_net_lp_return_bps']) for row in fee_split if int(row['dusk_total_base_fee_revenue_share_bps']) == share and row['routing_mode'] == route)
        fee_table[f'{share}:{route}'] = {
            'median_bps': statistics.median(values),
            'p10_bps': percentile(values, .10),
            'win_fraction': sum(value > 0 for value in values) / len(values),
        }
fee_table

In [ ]:
# Compact chart-ready rows used by the HTML report.
[{'share_bps': key.split(':')[0], 'routing': key.split(':')[1], **values} for key, values in fee_table.items()]

## Controller value, independently selected CPMM, and matched-policy ablation

In [ ]:
{'controller_counterfactual': summary['controller_counterfactual'], 'cpmm_product_selection_policy': summary['cpmm_product_selection_policy'], 'cpmm_product_fallback': summary['cpmm_product_fallback'], 'cpmm_matched_policy_ablation': summary['cpmm_matched_policy_ablation']}

## Mechanical decision

The benchmark does not convert a favorable median into a universal winner. Every fee-adjusted predicate must pass, including route- and flow-route robustness versus the independently synthetic-selected Dusk CPMM product comparator. The common Curve–DH2 rho governs only the comparative lock; Dusk's independent rho alone governs Dusk synthetic product readiness; CPMM's rho qualifies only CPMM. Marginal-price tracking defines stale readiness, while finite-size selected-order cost and the raw 1%-TVL depth-degradation probe remain service diagnostics. The matched-policy CPMM is a concentration ablation only and never enters winner gates. Gross results remain diagnostics for the fee-adjusted-selected profiles.

In [ ]:
decision = summary['decision']
{
    'Dusk fee-adjusted universal gate': decision['fee_adjusted']['dusk']['predicates'],
    'Curve fee-adjusted universal gate': decision['fee_adjusted']['curve']['predicates'],
    'Neighborhood common-envelope eligibility': decision['neighborhood_service_eligibility'],
    'Gross diagnostic scope': decision['gross_outcome_for_fee_adjusted_selected_profiles']['scope'],
}

In [ ]:
{
    'fee_adjusted_universal_native_winner': decision['fee_adjusted_universal_native_winner'],
    'gross_outcome_for_fee_adjusted_selected_profiles': decision['gross_outcome_for_fee_adjusted_selected_profiles'],
    'rule': decision['rule'],
}